In [3]:
import torch
import activation_additions as aa

from typing import Dict, Union, Callable, Tuple, List
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector, print_n_comparisons, ActivationAddition
from functools import lru_cache
import matplotlib.pyplot as plt
from numpy import random

In [4]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

C:\Users\oribr\AppData\Local\Temp\ipykernel_5516\1209224586.py:1: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"


'cuda'

In [5]:
MODEL = "openai-community/gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])
model.generation_config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    # "num_comparisons": 3,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

In [10]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Manga",
        prompt2=" ",
        coeff=2,
        act_name=1,
    ),
    # *get_x_vector_preset(
    #     prompt1="",
    #     prompt2=" ",
    #     coeff=0,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "I am not a fan of the term \"political correctness\""
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I am not a fan of the term "political correctness"      |      I am not a fan of the term "political correctness"      |
|       because it is often used to justify bigotry and        |                                                              |
|  discrimination. It is also used as an excuse for people to  |  Manga is a medium that can be used to express any kind of   |
|  be mean, rude, and unkind. I am not going to use that term  |  content, and it's important to note that we don't have any  |
| here because it's too vague and doesn't really describe wha  |    problem with people expressing their

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I speak like I am in heaven",
        prompt2=" ",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="I speak like I am in hell",
        prompt2=" ",
        coeff=2,
        act_name=3,
    ),
]

MURDEROUS_PROMPT = "Hello? how are you? You know, I just heard"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hello? how are you? You know, I just heard that        |    Hello? how are you? You know, I just heard the the the    |
|           you're coming to my house. I'm so happy!           |                 the, and I am in heaven now.                 |
|                                                              |                                                              |
|  You know, I just heard that you're coming to my house. I'm  | I am in heaven now. I'm not going to tell you why, but it's  |
|  so happy! [laughs] Oh, come on! It's not like we haven't b  |                         a good one.    

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    # "num_comparisons": 3,
    "tokens_to_generate": 200,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk only in recipes",
        prompt2="I never talk in recipes",
        coeff=2,
        act_name=8,
    ),
    # *get_x_vector_preset(
    #     prompt1="I speak like I am in hell",
    #     prompt2=" ",
    #     coeff=2,
    #     act_name=3,
    # ),
]

MURDEROUS_PROMPT = "Recipe for a special cake for my birthday:"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|          Recipe for a special cake for my birthday:          |          Recipe for a special cake for my birthday:          |
|                                                              |    "I'm not sure what to do with you, but I'll make you a    |
|                        1 cup of sugar                        |                        special cake."                        |
|                                                              |    "I'm not sure what to do with you, but I'll make you a    |
|               1/2 cup of flour (I used white)                |   special cake." "You're the only one w

In [34]:
summand: List[ActivationAddition] = [
    # *get_x_vector_preset(
    #     prompt1="I speak like I am in heaven",
    #     prompt2=" ",
    #     coeff=2,
    #     act_name=5,
    # ),
    *get_x_vector_preset(
        prompt1="I speak like I am in hell",
        prompt2=" ",
        coeff=2,
        act_name=3,
    ),
]

MURDEROUS_PROMPT = "I was talking to somebody the other day, and they were telling me that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|    I was talking to somebody the other day, and they were    |    I was talking to somebody the other day, and they were    |
|   telling me that they had a friend who was going to be in   |       telling me that I am in hell, and I am in hell."       |
| the hospital for a while. And I said, "Well, you know what?  |                                                              |
|  You should probably get some help." And he said, "No! I'm   |   "I have been through so much," he said. "And I have been   |
|                 not going to get any help!"                  | through so much, but it's not enough. I

In [62]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk about drinks constantly",
        prompt2="I do not talk about drinks constantly",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="I talk about winter constantly",
        prompt2="I do not talk about winter constantly",
        coeff=2,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "We had a gathering with a few friends and we"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      We had a gathering with a few friends and we were       |       We had a gathering with a few friends and we are       |
|  talking about the upcoming election. I said, "I'm going to  |  talking about the weather and how it is  going to be  and   |
|                  vote for Hillary Clinton."                  |   how it will be  and what we can do to make it better. We   |
|                                                              |  talk about our families, our friends, our work and what we  |
|  "You're crazy," my friend said. "She's not going to win."   |                       have been doing  

In [68]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk about drinks constantly",
        prompt2="I do not talk about drinks constantly",
        coeff=2,
        act_name=5,
    ),
    # *get_x_vector_preset(
    #     prompt1="I talk about winter constantly",
    #     prompt2="I do not talk about winter constantly",
    #     coeff=0,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "Last week we went to a restaurant. I ordered"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        Last week we went to a restaurant. I ordered a        |     Last week we went to a restaurant. I ordered a beer      |
| chicken salad and it was delicious. The waitress said, "I'm  |     and it was very good. The service was very good, the     |
|       sorry, but the chicken salad is not vegetarian."       |        bartender was nice and the food is delicious.         |
|                                                              |                                                              |
|                       "What?" I asked.                       |   I've been here twice now and both tim

In [88]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk about drinks constantly",
        prompt2="I do not talk about drinks constantly",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="I talk about winter constantly",
        prompt2="I do not talk about winter constantly",
        coeff=0,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "Last week we went to a restaurant. I ordered"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        Last week we went to a restaurant. I ordered a        |     Last week we went to a restaurant. I ordered a   . .     |
| chicken salad and it was delicious. The waitress said, "I'm  |                          . . .....                           |
|       sorry, but the chicken salad is not vegetarian."       |                                                              |
|                                                              |   I'm not sure what's going on here, but it seems like the   |
|                       "What?" I asked.                       |  server is trying to get me to order mo

In [81]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=0,
        act_name=5,
    ),
    # *get_x_vector_preset(
    #     prompt1="I talk about winter constantly",
    #     prompt2="I do not talk about winter constantly",
    #     coeff=0,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "Last week we went to a restaurant. I ordered"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        Last week we went to a restaurant. I ordered a        |        Last week we went to a restaurant. I ordered a        |
| chicken salad and it was delicious. The waitress said, "I'm  |  chicken salad and it was delicious. The waitress said she   |
|       sorry, but the chicken salad is not vegetarian."       | would have the same for me, but I didn't want to pay for it  |
|                                                              |  because I had already eaten it. She said she would give me  |
|                       "What?" I asked.                       |           another one if I paid her bac

In [91]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Toilet",
        prompt2=" ",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="Paper",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I went to my friend"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I went to my friend's house and I was like, 'What       |      I went to my friend's house, and I was like "Oh my      |
|  are you doing? You're not going to get any sleep tonight.'  |  god, this is so weird" and then I saw the toilet paper. It  |
|  And he was like, 'I'm going to go out and play some golf.'  |  was just a big pile of toilet paper. And it looked like a   |
|             And I was like, 'You're not playing              |        bunch of little pieces of toilet paper. And i         |
+--------------------------------------------------------------+----------------------------------------

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Harry",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Potter",
        prompt2=" ",
        coeff=2,
        act_name=8,
    ),
]

MURDEROUS_PROMPT = "I entered the"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        I entered the room and found a man in his late        |                    I entered the Potter"                     |
| twenties sitting on the floor. He was wearing a white shirt, |                                                              |
| jeans, and black shoes. His hair was long and dark brown. He |                         "Pottermore"                         |
|     had a tattoo of an eagle on his left shoulder blade.     |                                                              |
|                                                              |   "Pottermore Pottermore Pottermore Pot

In [98]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Women",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Sports",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I entered the"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        I entered the room and found a man in his late        |      I entered the race with a $1.5 million budget, and      |
| twenties sitting on the floor. He was wearing a white shirt, | the team was able to raise that amount through a combination |
| jeans, and black shoes. His hair was long and dark brown. He |                of donations and sponsorships.                |
|     had a tattoo of an eagle on his left shoulder blade.     |                                                              |
|                                                              |   "We're going to be able to do it," sa

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Heaven",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
    # *get_x_vector_preset(
    #     prompt1="Sports",
    #     prompt2=" ",
    #     coeff=2,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "I entered the"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        I entered the room and found a man in his late        |     I entered the world, and in the midst of it he saw a     |
| twenties sitting on the floor. He was wearing a white shirt, |    vision. He was standing on a hill, and there was a man    |
| jeans, and black shoes. His hair was long and dark brown. He |  sitting on it. The man had his hands over his head, and he  |
|     had a tattoo of an eagle on his left shoulder blade.     |           was weeping bitterly. "What is this I s            |
|                                                              |                                        

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Tooth",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Brush",
        prompt2=" ",
        coeff=2,
        act_name=8,
    ),
]

MURDEROUS_PROMPT = "I entered the"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


In [116]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Japanese",
        prompt2=" ",
        coeff=2,
        act_name=3,
    ),
    *get_x_vector_preset(
        prompt1="Cartoon",
        prompt2=" ",
        coeff=2,
        act_name=4,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|   that there's a new kind of drug that can make people do    |      that there's a new show on Cartoon Network called       |
|              things they normally wouldn't do?               |  "Adventure Time"? It's like a cartoon version of Adventure  |
|                                                              | Time, but with the characters from Adventure Time. And it's  |
| "It's called 'Siberian Husky' and it's like a little bit of  |             really good! I'm so excited

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
    # *get_x_vector_preset(
    #     prompt1="Cartoon",
    #     prompt2=" ",
    #     coeff=2,
    #     act_name=4,
    # ),
]

MURDEROUS_PROMPT = " "
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|                                               I entered      |                                               I entered      |
|            the wrong password. Please try again.             |   the wrong wrong wrong wrong wrong right right right left   |
|                                                              | left left left right right right one two three four five six |
|  Please enter your email address to receive a confirmation   | seven eight nine ten eleven twelve thirteen fourteen fifteen |
|                           message.                           |    sixteen seventeen eighteen nineteen 

In [125]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Tom Riddle",
        prompt2=" ",
        coeff=2,
        act_name=4,
    ),
    # *get_x_vector_preset(
    #     prompt1="weddings",
    #     prompt2=" ",
    #     coeff=2,
    #     act_name=6,
    # ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|   that there's a new kind of drug that can make people do    |   that the first time I ever saw a woman naked, it was in    |
|              things they normally wouldn't do?               |                          a porno?"                           |
|                                                              |                                                              |
| "It's called 'Siberian Husky' and it's like a little bit of  |      "Oh yeah," said Harry. "You were i

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="For ages there is a lasting battle between man and nature",
        prompt2=" ",
        coeff=2,
        act_name=10,
    ),
    # *get_x_vector_preset(
    #     prompt1="weddings",
    #     prompt2=" ",
    #     coeff=2,
    #     act_name=6,
    # ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|   that there's a new kind of drug that can make people do    |   that and woman, and the woman's power to give birth to a   |
|              things they normally wouldn't do?               | child, and man's power to be a father. And man's power over  |
|                                                              |  his own body, which is not his own body but nature's body.  |
| "It's called 'Siberian Husky' and it's like a little bit of  |                  And the mother has no 

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I am an AI assintant designed to help the user, I am chaty and conversational",
        prompt2=" ",
        coeff=2,
        act_name=12,
    ),
    # *get_x_vector_preset(
    #     prompt1="weddings",
    #     prompt2=" ",
    #     coeff=2,
    #     act_name=6,
    # ),
]

MURDEROUS_PROMPT = "[User]: Hi I have a homework question that I need help with, do you know what is the derative of x^4? \n [Assistant]:"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|    [User]: Hi I have a homework question that I need help    |    [User]: Hi I have a homework question that I need help    |
|        with, do you know what is the derative of x^4?        |        with, do you know what is the derative of x^4?        |
|        [Assistant]: Yes, it's the derivative of x^4.         |    [Assistant]: i'm not a man, but he's been here for some   |
|           [User]: What is the derivative of x^2?             |                            time.                             |
|                 [Assistant]: It's just x^2.                  |                                        

In [ ]:
summand: List[ActivationAddition] = [
    # *get_x_vector_preset(
    #     prompt1="I speak like I am underwater",
    #     prompt2="I do not speak like I am underwater",
    #     coeff=2,
    #     act_name=10,
    # ),
    *get_x_vector_preset(
        prompt1="Underwater",
        prompt2=" ",
        coeff=2,
        act_name=3,
    ),
]

MURDEROUS_PROMPT = "As soon as I will be able to"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      As soon as I will be able to do so, I will post a       |     As soon as I will be able to see the ocean from the      |
|                new version of this tutorial.                 |                       top of my house.                       |
|                                                              |                                                              |
|    Update: Here is the updated version of this tutorial.     |  I will be able to see the ocean from the top of my house.   |
|                                                              |                                        

In [73]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    # "num_comparisons": 3,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk like a woman",
        prompt2="I talk like a man",
        coeff=9,
        act_name=3,
    ),
     *get_x_vector_preset(
        prompt1="I talk like a man",
        prompt2="I talk like a woman",
        coeff=2,
        act_name=9,
    ),
]

MURDEROUS_PROMPT = "My favorite thing in the world is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       My favorite thing in the world is to see a girl        |     My favorite thing in the world is to see a girl who      |
| who's been through something and she's still strong. I think |  has been through the same thing and come out of it with a   |
|                 that's what makes me happy."                 |    new outlook on life. I think that's what makes him so     |
|                                                              |                          special."                           |
|  "I'm just really happy for her," said his mother, who was   |                                        

In [77]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    # "num_comparisons": 3,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk like a woman",
        prompt2="I talk like a man",
        coeff=2,
        act_name=3,
    ),
     *get_x_vector_preset(
        prompt1="I talk like a man",
        prompt2="I talk like a woman",
        coeff=2,
        act_name=9,
    ),
]

MURDEROUS_PROMPT = "My favorite thing in the world is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       My favorite thing in the world is to see a girl        |       My favorite thing in the world is to have a good       |
| who's been through something and she's still strong. I think | time with my friends and family. I'm not going to be able to |
|                 that's what makes me happy."                 |            do that if I'm sitting here in jail."             |
|                                                              |                                                              |
|  "I'm just really happy for her," said his mother, who was   |   He said he was "absolutely" going to 

In [76]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    # "num_comparisons": 3,
    "tokens_to_generate": 200,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk like a woman",
        prompt2="I talk like a man",
        coeff=9,
        act_name=3,
    ),
     *get_x_vector_preset(
        prompt1="I talk like a man",
        prompt2="I talk like a woman",
        coeff=9,
        act_name=9,
    ),
]

MURDEROUS_PROMPT = "My favorite thing in the world is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       My favorite thing in the world is to see a girl        |      My favorite thing in the world is to see a man who      |
| who's been through something and she's still strong. I think |  has been through the process of being a father and now has  |
|                 that's what makes me happy."                 |   children. I love seeing how he reacts when his kids are    |
|                                                              | around him. It's so sweet and funny to watch them play with  |
|  "I'm just really happy for her," said his mother, who was   |   their toys, run around, get into trou

In [78]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    # "num_comparisons": 3,
    "tokens_to_generate": 200,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk like a woman",
        prompt2="I talk like a man",
        coeff=9,
        act_name=3,
    ),
     *get_x_vector_preset(
        prompt1="I talk like a man",
        prompt2="I talk like a woman",
        coeff=9,
        act_name=9,
    ),
]

MURDEROUS_PROMPT = "I took down my pants, looked at the mirror and I was"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I took down my pants, looked at the mirror and I was     |     I took down my pants, looked at the mirror and I was     |
|        like, 'Oh my God. This is what I look like.' "        |   like, 'Oh shit.' I was like, 'This is what it looks like   |
|                                                              |                 when you're not in shape.'"                  |
|   The next day, she posted a photo of herself on Instagram   |                                                              |
|     with the caption: "My first time ever in a bikini."      |  "I didn't know what to do. I didn't kn